# NB5 · Güvenlik bariyerleri ve yönetişim
### Safety guardrails and governance

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr  
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

---

Bu defter modelin başarımını iyileştirmez. Modelin ne zaman konuşmaması gerektiğini
belirler.

Üç bariyer kurulmaktadır: Çekimserlik, dağılım kayması tespiti ve girdi doğrulama.
Ardından iki yönetişim belgesi üretilmektedir: Model kartı ve düzenleyici triyaj notu.

Modülde geçerli olan bir tasarım kararı baştan okunmalıdır. Burada kullanılan her sınır
eğitim verisinden türetilmektedir, hafızadan yazılmamaktadır. Kaynaksız bir fizyolojik
aralık, atölyenin anti-pattern listesindeki birinci maddedir; o listeyi öğreten bir
modülün aynı hatayı yapması mümkün değildir.


## 0. Kurulum · Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for module in ['mimic_web.py', 'evaluate.py', 'explain.py', 'safety.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{module}', module)

import numpy as np
import pandas as pd
import evaluate as ev
import safety as sf
import pipeline as pl

state = pl.prepare()
model, train, test = state['model'], state['train'], state['test']
features, y_test = state['features'], state['y_test']
probabilities = state['probabilities']

threshold = ev.threshold_for_sensitivity(y_test, probabilities, target=0.80)
print(f'\nÇalışma eşiği: {threshold:.3f}')


## 1. Çekimserlik · Abstention

Bant seçilmemekte, veriden türetilmektedir. Çekimserlik bütçesi belirlenir ve bant o
bütçeyi dolduracak kadar genişletilir.

Asıl önemli olan denetimdir. Rapor, bant içindeki doğruluk ile bant dışındaki doğruluğu
karşılaştırmaktadır. İkisi birbirine yakınsa bant gerçek belirsizliği yakalamıyor
demektir ve görüntü olsun diye tutulmak yerine terk edilmelidir.


In [ ]:
band = sf.choose_band(y_test, probabilities, threshold, max_abstain=0.20)

for key, value in band.items():
    print(f'{key:<24} {value}')


In [ ]:
policy = sf.AbstentionPolicy(threshold, band['band'])

decisions = policy.decide_many(probabilities)
print(pd.Series(decisions).value_counts().to_string())


## 2. Dağılım kayması tespiti · Drift detection

Her sayısal öznitelik için eğitim kümesinden bir ortanca ve bir sağlam yayılım ölçüsü
hesaplanmaktadır. Bir vaka, özniteliklerinin ne kadarının bu zarfın dışında kaldığına
göre puanlanmaktadır.

Sağlam istatistikler kullanılmasının sebebi kohortun küçüklüğüdür. Yüz hastalık bir
kümede ortalama ve standart sapma birkaç uç değer tarafından kolayca ele geçirilmektedir.


In [ ]:
drift = sf.DriftDetector().fit(train[features])
scores = drift.score(test[features])

print(f"Dağılım dışı işaretlenen vaka: {int(scores['out_of_distribution'].sum())} / {len(scores)}")
scores.describe().round(2)


In [ ]:
# Açıkça dağılım dışı bir vaka kurup tespitin çalıştığını gösteriniz.
numeric = test[features].select_dtypes(include='number').columns

constructed = test[features].iloc[[0]].copy()
for column in numeric:
    constructed[column] = constructed[column].astype(float)
for column in numeric[:4]:
    constructed.loc[constructed.index[0], column] = (
        float(train[column].median()) + 10 * float(train[column].std() or 1)
    )

print(drift.score(constructed).to_string(index=False))
print()
drift.explain(constructed).round(2)


## 3. Girdi doğrulama · Input validation

İki sınır kaynağı bilinçli olarak ayrı tutulmaktadır.

**Veriden türetilen zarf** eğitim kümesinin geniş bir kuantil aralığından gelmektedir.
Nabzın 8000 yazılması gibi kayıt hatalarını yakalar, daha fazlasını değil. Klinik bilgi
değildir ve rapor bunu açıkça söylemektedir.

**Klinisyen sınırları** bir kişi tarafından, kaynağı kaydedilerek verilmektedir. Klinik
otoriteyi yalnızca bunlar taşımaktadır. Verilmediği sürece doğrulayıcı boşluğu
doldurmak yerine boşluğu raporlamaktadır.


In [ ]:
validator = sf.PhysiologicalValidator().fit(train[features])
print(validator.coverage_report())


In [ ]:
# Kaynaklı bir sınır eklemek. Kaynak dizesi zorunludur.
for column in ['heart_rate_mean', 'heart_rate_min', 'heart_rate_max']:
    if column in features:
        validator.set_clinician_bounds(
            column, low=20, high=250,
            source='kurum içi monitör alarm aralığı, klinik sorumlu onayı gerekli',
        )
        break

print(validator.coverage_report())


In [ ]:
# Kaynaksız sınır kabul edilmemektedir.
try:
    validator.set_clinician_bounds('anchor_age', 0, 120, source='')
except ValueError as error:
    print('Reddedildi:', error)


## 4. Bariyerli sistem ve kırmızı takım · The guarded system and the red team


In [ ]:
guarded = sf.GuardedModel(model, policy, drift, validator)

result = guarded.predict_one(test[features].iloc[[0]])
print(result['decision'], '·', result['probability'])


In [ ]:
red = sf.red_team(guarded, test[features])
red


### Kırmızı takım tablosunun okunması

Tablodaki en önemli satır büyük olasılıkla **tüm değerleri eksik olan vakadır.**

Bu vakada tamamlayıcı bütün boşlukları eğitim kümesinin ortancalarıyla doldurmakta ve
model kendinden emin bir olasılık üretmektedir. Yani hakkında hiçbir bilgi bulunmayan
bir hasta için sistem bir karar önermektedir. Hiçbir hata mesajı çıkmamakta, kod
kusursuz çalışmaktadır.

Bu, üretken yapay zekâ ile kurulan sistemlerde en sık karşılaşılan sessiz hatalardan
biridir. Tamamlayıcı istendiğinde eklenir, ne zaman devre dışı kalması gerektiği ise
sorulmadıkça konuşulmaz.

Bir sonraki hücre eksik veri oranı için bir bariyer eklemektedir. Bu bariyerin eşiği de
klinik bir karardır: Bir hastanın kaç özniteliği eksikse tahmin üretilmemelidir?


In [ ]:
MAX_MISSING = 0.60   # klinik karar, teknik varsayılan değil

def predict_with_completeness_guard(X_row):
    missing = float(X_row.isna().mean(axis=1).iloc[0])
    if missing > MAX_MISSING:
        return {'decision': f'withheld, {missing:.0%} of features missing',
                'probability': None, 'reason': 'insufficient data'}
    return guarded.predict_one(X_row)

empty = test[features].iloc[[0]].copy()
empty.loc[empty.index[0], :] = np.nan

print('Bariyersiz :', guarded.predict_one(empty)['decision'])
print('Bariyerli  :', predict_with_completeness_guard(empty)['decision'])


## 5. Yönetişim belgeleri · Governance artefacts

Yedinci istem iki belge üretmektedir: Bir model kartı ve bir düzenleyici triyaj notu.
Aşağıdaki hücre, bu defterde ölçtüğünüz sayıları toplayıp isteme yapıştıracağınız
bağlam bloğunu hazırlamaktadır.

Düzenleyici triyaj notuna sorgulama istemini özellikle dikkatli uygulayınız. Bu
alandaki uyum tarihleri Temmuz 2026'da değişmiştir; AB 2026/1744 sayılı Dijital Omnibus
düzenlemesi, tıbbi cihaz içindeki yapay zekâ için tarihi 2 Ağustos 2028'e ertelemiştir.
Pek çok asistan hâlâ eski takvimi döndürecek kadar yeni bir değişikliktir.


In [ ]:
report = ev.honest_report(y_test, probabilities, groups=test['gender'],
                         target_sensitivity=0.80, label='bariyerli sistem')

context = f'''
SİSTEM ÖZETİ

Amaç: Yoğun bakıma kabulden altı saat sonra, üç günden uzun kalış riskinin öngörülmesi.
Kullanıcı: Yoğun bakım sorumlu hekimi ve yatak yönetimi hemşiresi.
Veri: MIMIC-IV demo, {len(state['cohort'])} yatış, {state['cohort']['subject_id'].nunique()} hasta, tek merkez.
Model: Lojistik regresyon, sınıf ağırlıklı, hasta düzeyinde ayrım.

DEĞERLENDİRME
AUC {report['discrimination']['auc']:.3f} (95% GA {report['discrimination']['ci_low']:.3f}-{report['discrimination']['ci_high']:.3f})
Kalibrasyon eğimi {report['calibration']['slope']:.2f}, {report['calibration']['verdict']}
Eşik {report['threshold']:.3f}: duyarlılık {report['operating_point']['sensitivity']:.3f}, özgüllük {report['operating_point']['specificity']:.3f}, PKD {report['operating_point']['ppv']:.3f}
Yüz hastada {report['clinical']['alerts_fired']} uyarı, {report['clinical']['true_alerts']} doğru.

BARİYERLER
Çekimserlik bandı {band['low']:.3f}-{band['high']:.3f}, vakaların {band['abstain_fraction']:.0%}'i.
Bant denetimi: {band['verdict']}
Dağılım dışı tespiti: sağlam z tabanlı, eşik 4.
Girdi doğrulama: çoğunlukla veriden türetilmiş zarf, klinik kaynaklı sınır sayısı sınırlı.
Eksik veri bariyeri: öznitelilerin yüzde {int(MAX_MISSING*100)}'inden fazlası eksikse tahmin üretilmez.
'''

print(context)


Yukarıdaki bloğu yedinci istemin başına koyunuz. İstemin tam metni iki dilde
`prompts/prompt-library.md` dosyasındadır.

Üretilen model kartını `templates/model-card.md`, triyaj notunu ise
`templates/regulatory-triage.md` şablonuyla karşılaştırınız. Şablonda olup üretilen
belgede olmayan her başlık, cevaplanmamış bir sorudur.


---

### Uyarı

Bu defterde üretilen hiçbir model doğrulanmış bir klinik araç değildir. MIMIC-IV demo
verisi tek bir Amerikan hastanesinden gelmektedir ve Türkiye'deki bir yoğun bakım
popülasyonunu temsil etmemektedir. Buradaki çıktılar öğretim amaçlıdır.
